# edit v6.1 pilot · 题目与 QIB 判分审阅（20 题）

- 题库：`data/synth_v61_pilot/questions.jsonl`（image-first 出题协议 v6.1；严格校验 PASS 20 / WARN 0 / REJECT 0）
- 判分协议：`benchmark/edit/codex_score_prompt_edit_v2.md`（QIB 三档 **0 / 60 / 100**，三维官方钳制 d2/d3 ≤ d1，`official_total` 为钳制后均值）
- Gemini：`scores_qib/gemini/scores.jsonl`（judge = gpt-5.6-sol-built-in，已完成）＋ 旧 1–5 诊断分 `scores/codex_blind/scores.jsonl`（同判官，哈希核验 20/20 为 Gemini 输出）
- Bagel：`scores_qib/bagel/scores.jsonl`（盲评落盘后**重跑本 notebook 自动补全**；未落盘时卡片只展示图与占位。注：旧 1–5 与新 QIB 两轮盲分均只评过 Gemini，Bagel 从未评分）
- 配对：`scores_qib/paired/`（两模型分数齐备后由 `eval_codex_score.py compare` 产出）
- ⚠️ 旧 1–5 与 QIB 0/60/100 是两套口径，并列仅作对照，**禁止混用或线性换算**

> Cell 1 数据与总览 ｜ Cell 2 逐题卡（题面 + 出题要素 + BEFORE/两模型 AFTER + Gemini 双口径逐维打分同行展示）｜ Cell 3 分组透视与配对
> 纯只读；Run All 即可。

In [ ]:
\
# ---- v6.1 pilot · 数据加载与总览 ----
import json
from pathlib import Path

import pandas as pd

BASE = Path('data/synth_v61_pilot')
if not (BASE / 'questions.jsonl').exists():
    BASE = Path('/tank/demiwtg/benchmark/edit/data/synth_v61_pilot')
EDIT_DATA = BASE.parent


def _jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.open(encoding='utf-8') if l.strip()] if p.exists() else []


def _json(p):
    p = Path(p)
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None


QS = {q['qid']: q for q in _jsonl(BASE / 'questions.jsonl')}
GEM = {r['qid']: r for r in _jsonl(BASE / 'scores_qib/gemini/scores.jsonl')}
OLD = {r['qid']: r for r in _jsonl(BASE / 'scores/codex_blind/scores.jsonl')}
BAG = {r['qid']: r for r in _jsonl(BASE / 'scores_qib/bagel/scores.jsonl')}
GEM_REP = _json(BASE / 'scores_qib/gemini/report.json')
OLD_REP = _json(BASE / 'scores/codex_blind/report.json')
BAG_REP = _json(BASE / 'scores_qib/bagel/report.json')
PAIRED_REP = _json(BASE / 'scores_qib/paired/report.json')
PAIRED = {r['qid']: r for r in _jsonl(BASE / 'scores_qib/paired/paired_scores.jsonl')}

_gdir = next((BASE / 'gemini/imgs').glob('*'), None) if (BASE / 'gemini/imgs').exists() else None
IMG = {
    'before': lambda q: EDIT_DATA / q['_sample_image'],
    'gemini': lambda q: _gdir / f"{q['qid']}.png" if _gdir else None,
    'bagel': lambda q: BASE / 'bagel/imgs' / f"{q['qid']}.png",
}

assert QS, f'题库未找到：{BASE / "questions.jsonl"}'
print(f"题库 {len(QS)} 题 ｜ Gemini QIB {len(GEM)} ｜ Gemini 旧1-5 {len(OLD)}"
      f" ｜ Bagel 分数 {len(BAG)} ｜ paired {len(PAIRED)}")
if GEM_REP:
    print(f"Gemini QIB overall {GEM_REP['overall']}", end='')
if OLD_REP:
    print(f" ｜ Gemini 旧1-5 overall {OLD_REP['overall']}/5", end='')
if BAG_REP:
    print(f" ｜ Bagel overall {BAG_REP['overall']}")
else:
    print('\n（Bagel 从未盲评：两轮盲分（旧1-5 与 QIB）哈希核验均为 Gemini 输出；'
          'Bagel 落盘后重跑本 notebook 自动补全）')

comp = pd.DataFrame({
    '字段': ['level', 'suite', '_batch', 'edit_type'],
}).iloc[0:0]
rows = []
for f in ('level', 'suite', '_batch', 'edit_type'):
    vc = pd.Series([q.get(f, '?') for q in QS.values()]).value_counts()
    for k, n in vc.items():
        rows.append({'字段': f, '取值': k, 'n': n})
display(pd.DataFrame(rows))

In [ ]:
\
# ---- v6.1 pilot · 逐题审阅卡：题面 + 出题要素 + 三图 + QIB 逐维打分 ----
import base64
import io

from IPython.display import HTML, display
from PIL import Image

IMG_MAX = 720


def _img_uri(path, max_side=IMG_MAX):
    if not path or not Path(path).exists():
        return None
    im = Image.open(path).convert('RGB')
    w, h = im.size
    s = max_side / max(w, h)
    if s < 1:
        im = im.resize((round(w * s), round(h * s)), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()


def _fig(path, title, badge=''):
    uri = _img_uri(path)
    if uri is None:
        return (f'<figure style="margin:4px;text-align:center;flex:1">'
                f'<figcaption style="color:#999">（{title} 缺失）</figcaption></figure>')
    b = f'　<b style="color:#111">{badge}</b>' if badge else ''
    return (f'<figure style="margin:4px;text-align:center;flex:1;min-width:280px">'
            f'<img src="data:image/jpeg;base64,{uri}" style="max-width:{IMG_MAX}px;width:100%">'
            f'<figcaption style="font-size:12.5px;color:#555">{title}{b}</figcaption></figure>')


def _score_block(name, score):
    if score is None:
        return (f'<div style="margin:6px 0"><b>{name}</b>：'
                '<span style="color:#999">盲评未落盘（part_*.jsonl 就绪后重跑本 notebook 自动补全）</span></div>')
    v1 = score.get('schema') == 'edit-codex-v1'
    unit = '/5' if v1 else ''
    head = (f'<b>{name}</b>：official_total <b>{score["official_total"]}{unit}</b>'
            f'（validity={score["validity"]["status"]}，confidence={score.get("confidence", "?")}）')
    lines = [f'<div style="margin:6px 0">{head}</div>']
    for d in score['raw_dimensions']:
        if v1:
            s = d['score']
            mark = f'score {s}/5'
            color = '#c0392b' if s <= 2 else ('#1e7e34' if s >= 4 else '#8a6d3b')
        else:
            mark = f'tier{d["tier"]} → {d["mapped"]}'
            color = '#c0392b' if d['tier'] == 0 else ('#1e7e34' if d['tier'] == 2 else '#8a6d3b')
        lines.append(f'<div style="margin:2px 0 2px 14px"><b>d{d["key"][1]} {d["label"]}</b>：'
                     f'<span style="color:{color};font-weight:700">{mark}</span>｜{d["reason"]}</div>')
    cf = score.get('critical_failures') or []
    if cf:
        lines.append(f'<div style="margin:2px 0 2px 14px;color:#c0392b">critical_failures：{"；".join(cf)}</div>')
    return ''.join(lines)


def _badges(qid):
    out = []
    g = GEM.get(qid)
    o = OLD.get(qid)
    b = BAG.get(qid)
    parts = []
    if g:
        parts.append(f'QIB <b>{g["official_total"]}</b>')
    if o:
        parts.append(f'旧1-5 <b>{o["official_total"]}</b>')
    out.append(f'Gemini {"｜".join(parts) if parts else "—"}')
    out.append(f'Bagel <b>{b["official_total"] if b else "⏳"}</b>')
    if qid in PAIRED:
        p = PAIRED[qid]
        mark = {'left': 'G胜', 'right': 'B胜', 'tie': '平'}[p['winner']]
        out.append(f'Δ(G−B) <b>{p["delta_left_minus_right"]:+g}</b>（{mark}）')
    return '　'.join(out)


def _qmeta(q):
    lr = q.get('level_reason') or {}
    pairs = [
        ('定位', q.get('targeting_types')),
        ('必然后果', q.get('consequence_types')),
        ('保持义务', q.get('preservation_types')),
        ('场景复杂度', q.get('scene_types')),
        ('跳类型', q.get('hop_types')),
        ('知识类别', q.get('knowledge_categories')),
        ('弱点', q.get('weak_points')),
        ('义务计数', [f"{k}={v}" for k, v in lr.items() if isinstance(v, int)]),
    ]
    lis = ''.join(f'<li><b>{k}</b>：{"、".join(map(str, v or []))}</li>' for k, v in pairs if v)
    return f'<details style="margin:4px 0"><summary style="cursor:pointer;color:#567">出题要素（level_reason / 设计维度菜单）</summary><ul style="margin:4px 0 4px 8px">{lis}</ul></details>'


for qid in sorted(QS):
    q = QS[qid]
    html = [
        f'<h3 style="margin:18px 0 2px">{qid} · {q.get("edit_type")} · {q.get("level")} · '
        f'{q.get("suite", "basic")} · {q.get("_batch", "?")}　｜　{_badges(qid)}</h3>',
        f'<div style="margin:2px 0"><b>编辑指令</b>：{q["edit_instruction"]}</div>',
        _qmeta(q),
        '<div style="display:flex;flex-wrap:wrap;align-items:flex-start">',
        _fig(IMG['before'](q), 'BEFORE（源图）'),
        _fig(IMG['gemini'](q), 'Gemini AFTER',
             f'{GEM[qid]["official_total"]}' if qid in GEM else '未评'),
        _fig(IMG['bagel'](q), 'Bagel AFTER',
             f'{BAG[qid]["official_total"]}' if qid in BAG else '待盲评'),
        '</div>',
        _score_block('Gemini · QIB 0/60/100', GEM.get(qid)),
        _score_block('Gemini · 旧 1–5 诊断分（已废弃口径，仅对照）', OLD.get(qid)),
        _score_block('Bagel', BAG.get(qid)),
        '<hr style="margin:10px 0">',
    ]
    display(HTML(''.join(html)))

In [ ]:
\
# ---- v6.1 pilot · 分组透视（Gemini；Bagel/paired 落盘后自动并入） ----
rows = []
for key, label in (('by_level', 'level'), ('by_suite', 'suite'),
                   ('by_edit_type', 'edit_type'), ('by_source_batch', 'source_batch')):
    reps = [('Gemini', GEM_REP), ('Gemini旧(1-5)', OLD_REP), ('Bagel', BAG_REP)]
    groups = sorted({g for _, r in reps if r for g in r.get(key, {})})
    for g in groups:
        row = {'分组': label, '取值': g}
        for name, rep in reps:
            v = rep.get(key, {}).get(g) if rep else None
            row[f'{name}'] = v['official_total'] if v else None
        row['n'] = next((v['n'] for _, r in reps if r
                         for v in [r.get(key, {}).get(g)] if v), None)
        rows.append(row)
if rows:
    display(pd.DataFrame(rows).set_index(['分组', '取值']))

if PAIRED_REP:
    ov = PAIRED_REP['overall']
    print(f"paired（左=gemini-3.1-flash-image，右=BAGEL-7B-MoT）："
          f"G {ov['gemini-3.1-flash-image']} vs B {ov['BAGEL-7B-MoT']}，"
          f"Δ(G−B)={ov['delta_left_minus_right']}，"
          f"W/T/L={ov['wins_ties_losses']}")
    display(pd.DataFrame(list(PAIRED.values())).set_index('qid')[
        ['level', 'edit_type', 'suite', 'left_total', 'right_total',
         'delta_left_minus_right', 'winner']])
else:
    print('（paired 未产出：Bagel 盲评落盘并 compare 后重跑本 cell）')